In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

DATA_DIR  = '/content/drive/MyDrive/atnlyze/data'
MODEL_DIR = '/content/bert_waf'

# Extract model from zip if not already done
import zipfile
model_zip = '/content/drive/MyDrive/atnlyze/models/bert_waf.zip'
if not os.path.exists(MODEL_DIR):
    print("Extracting model...")
    with zipfile.ZipFile(model_zip, 'r') as z:
        z.extractall('/content')
    print("Done.")
else:
    print("Model already extracted.")


In [ ]:
!pip install -q "transformers>=4.41,<5" datasets accelerate scikit-learn


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')


In [ ]:
# Upload v2_test_bert.csv and v2_test_hard_bert.csv to MyDrive/atnlyze/data/
for f in ['v2_test_bert.csv', 'v2_test_hard_bert.csv']:
    path = os.path.join(DATA_DIR, f)
    print(f, ':', 'FOUND' if os.path.exists(path) else 'MISSING')


In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, classification_report
from tqdm import tqdm

BATCH_SIZE = 64
MAX_LENGTH = 128

device    = torch.device("cuda")
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_DIR)
model     = DistilBertForSequenceClassification.from_pretrained(MODEL_DIR)
model.to(device)
model.eval()
print("Model loaded on", device)


In [ ]:
def run_inference(texts):
    probs_all = []
    for i in tqdm(range(0, len(texts), BATCH_SIZE)):
        batch  = texts[i:i+BATCH_SIZE]
        inputs = tokenizer(batch, padding=True, truncation=True,
                           max_length=MAX_LENGTH, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=1)[:, 1]
        probs_all.extend(probs.cpu().numpy().tolist())
    return probs_all

def evaluate(df, text_col, label):
    texts  = df[text_col].tolist()
    labels = df["label"].tolist()
    probs  = run_inference(texts)
    preds  = [1 if p >= 0.5 else 0 for p in probs]

    acc              = accuracy_score(labels, preds)
    p, r, f1, _      = precision_recall_fscore_support(labels, preds, average="binary")
    try:
        roc = roc_auc_score(labels, probs)
        roc_str = f"{roc:.4f}"
    except ValueError:
        roc_str = "N/A (single class)"

    print(f"\n{'='*55}")
    print(f"  {label}")
    print(f"  Samples: {len(df)} (benign={sum(l==0 for l in labels)}, malicious={sum(l==1 for l in labels)})")
    print(f"{'='*55}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {p:.4f}")
    print(f"  Recall    : {r:.4f}")
    print(f"  F1 Score  : {f1:.4f}")
    print(f"  ROC AUC   : {roc_str}")
    print()
    print(classification_report(labels, preds, target_names=["benign", "malicious"]))
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}

def evaluate_by_source(df, text_col):
    for source in ["docker", "csic", "modsec"]:
        subset = df[df["source"] == source]
        if len(subset) == 0:
            continue
        evaluate(subset, text_col, f"Transformer - source: {source}")

def evaluate_by_attack_type(df, text_col):
    malicious = df[df["label"] == 1]
    print(f"\n{'='*55}")
    print(f"  Per-attack-type metrics (malicious only)")
    print(f"{'='*55}")
    print(f"  {'Type':<12} {'N':>6} {'Precision':>10} {'Recall':>8} {'F1':>8}")
    print(f"  {'-'*50}")
    for atype in sorted(malicious["attack_type"].unique()):
        subset = malicious[malicious["attack_type"] == atype]
        if len(subset) < 5:
            continue
        texts  = df.loc[subset.index, text_col].tolist()
        labels = subset["label"].tolist()
        probs  = run_inference(texts)
        preds  = [1 if p >= 0.5 else 0 for p in probs]
        p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
        print(f"  {atype:<12} {len(subset):>6} {p:>10.4f} {r:>8.4f} {f1:>8.4f}")


In [ ]:
# Full v2 test set
df = pd.read_csv(os.path.join(DATA_DIR, 'v2_test_bert.csv'))
normal_results = evaluate(df, "structured_text", "v2 test set (all sources)")
evaluate_by_source(df, "structured_text")
evaluate_by_attack_type(df, "structured_text")


In [ ]:
# ModSec OOD holdout
modsec = df[df["source"] == "modsec"]
evaluate(modsec, "structured_text", "ModSec OOD holdout (genuinely unseen)")


In [ ]:
# Adversarial hard test set
hard_df      = pd.read_csv(os.path.join(DATA_DIR, 'v2_test_hard_bert.csv'))
hard_results = evaluate(hard_df, "structured_text", "Adversarial hard test")
evaluate_by_attack_type(hard_df, "structured_text")


In [ ]:
# Summary - Normal vs Adversarial
print("\n" + "="*55)
print("  SUMMARY - Normal vs Adversarial")
print("="*55)
print(f"  {'Metric':<12} {'Normal':>10} {'Adversarial':>12} {'Gap':>8}")
print(f"  {'-'*46}")
for metric in ("accuracy", "precision", "recall", "f1"):
    nv  = normal_results[metric]
    hv  = hard_results[metric]
    gap = nv - hv
    print(f"  {metric:<12} {nv:>10.4f} {hv:>12.4f} {gap:>8.4f}")
f1_gap = normal_results["f1"] - hard_results["f1"]
print()
if f1_gap < 0.01:
    print("  F1 gap < 0.01 - transformer robust to evasion techniques.")
elif f1_gap < 0.05:
    print("  F1 gap < 0.05 - mild degradation. Acceptable for thesis.")
else:
    print(f"  F1 gap = {f1_gap:.4f} - notable degradation.")
